In [ ]:
from glob import glob

model_paths = glob('../models/Guillen2021/strains/*.xml')
model_ids = [x.split('_')[1][:-4] for x in model_paths]

In [ ]:
from reframed.io.cache import ModelCache

cache = ModelCache(model_ids, model_paths, load_args={'flavor': 'bigg'})
models = [cache.get_model(x) for x in model_ids]

In [ ]:
import pandas as pd
from reframed import Environment

media = pd.read_csv('../data/Guillen2021/generated/media_db.tsv', sep='\t')
M9 = Environment.from_compounds(list(media['compound']), max_uptake=1000)
M9['R_EX_glc__D_e'] = (-10, float('inf'))

In [ ]:
from steadiercom.steadiercom import SteadierCom, SteadierSample
from reframed import Community

In [ ]:
sols = []

for i in range(30):
    community = Community("test", sample(models, 10), copy_models=False)
    M9.apply(community.merged_model, inplace=True, exclusive=True, warning=False)
    new_sols = SteadierSample(community, n=30, growth=0.221, allocation=True)
    sols.extend(new_sols)

In [ ]:
data = {'Δ' + model.id.split('_')[1]: [sol_i.abundance.get(model.id) for sol_i in sols] for model in models}
data = pd.DataFrame(data).T

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1,1,figsize=(5,8))
data.mean(axis=1, skipna=True).sort_values().plot.barh(log=True, ax=ax)
ax.set_xlabel('relative abundance')
fig.tight_layout()
fig.savefig('ecoli_38_strains.png', dpi=150)